In [ ]:
                                                           
import sys, glob, subprocess, os

hits = sorted(glob.glob('/kaggle/input/**/exp/exp002/predict.py', recursive=True))
assert hits, 'exp/exp002/predict.py  Kaggle Dataset  /kaggle/input '
CODE_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))                                            
print('CODE_ROOT =', CODE_ROOT)

def _zarr_major():
    try:
        import zarr
        return int(zarr.__version__.split('.')[0]), zarr.__version__
    except Exception as e:
        return -1, str(e)

major, ver = _zarr_major()
print('zarr (before):', ver)
if major < 3:
    wh = os.path.join(CODE_ROOT, 'wheels')
    assert os.path.isdir(wh), f'zarr {ver}  v3 。{wh}/  zarr>=3  wheel(cp312) '
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index',
                           '--find-links', wh, 'zarr>=3'])
    for m in [k for k in list(sys.modules) if k == 'zarr' or k.startswith('zarr.')]:
        del sys.modules[m]
    major, ver = _zarr_major()
print('zarr (final):', ver)
assert major >= 3, 'zarr v3 '

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
                                                    
                                                                                
                                                
                                                
if not torch.cuda.is_available():
    print('!! WARNING: cuda 。CPU 。', flush=True)

In [ ]:
                                   
import glob, json as _json, os, shutil

COMP_SLUG = 'biohub-cell-tracking-during-development'
_cands = {p for p in glob.glob(f'/kaggle/input/**/{COMP_SLUG}', recursive=True) if os.path.isdir(p)}
if os.path.isdir(f'/kaggle/input/{COMP_SLUG}'):
    _cands.add(f'/kaggle/input/{COMP_SLUG}')
comp_cands = sorted(_cands)
assert comp_cands, f' input : {COMP_SLUG}'
COMP = comp_cands[0]
TEST_DIR = os.path.join(COMP, 'test')
if not os.path.isdir(TEST_DIR):
    TEST_DIR = COMP
n_zarr = len(glob.glob(os.path.join(TEST_DIR, '**', '*.zarr'), recursive=True))
print('TEST_DIR =', TEST_DIR, '| .zarr datasets =', n_zarr)
assert n_zarr > 0, f'.zarr : {TEST_DIR}'
                                                                      
                                                                        
                                             
                                                                    
                                       
if n_zarr > 200:
    print(f'!! WARNING: .zarr  {n_zarr} 。TEST_DIR  '
          f'train : {TEST_DIR}', flush=True)

                                                                       
                                                                                    
                        
WEIGHT_KEYS = ['biohub-temporal-unet3d-seed314159-v1', 'biohub-tracking-support-pack-50ep-v1']

_pth = sorted(glob.glob('/kaggle/input/**/edge_predictor_best.pth', recursive=True))
assert _pth, 'edge_predictor_best.pth '
print(':')
for q in _pth:
    print('  ', q)

WEIGHT_DIRS = []
for k in WEIGHT_KEYS:
    hit = [q for q in _pth if k in q]
    assert hit, f'{k}  Add Input 。: {_pth}'
    src = hit[0]
    ocfg_p = os.path.join(os.path.dirname(src), 'config.json')
    ocfg = _json.loads(open(ocfg_p).read()) if os.path.exists(ocfg_p) else {}
    d = f'/kaggle/working/w_{k[:24].replace("-", "_")}'
    os.makedirs(d, exist_ok=True)
    shutil.copy(src, os.path.join(d, 'best.pt'))
    open(os.path.join(d, 'model_config.json'), 'w').write(_json.dumps({
        'unet_out_channels': ocfg.get('unet_out_channels', 32),
        'unet_layers': ocfg.get('unet_layers', [32, 64, 128]),
        'hidden_dim': 128, 'n_heads': 4, 'n_blocks': 4, 'dropout': 0.3,
        'downsample': ocfg.get('downsample', [1, 4, 4]),
        'window_size': ocfg.get('window_size', 2),
        'pool_kernel_um': ocfg.get('pool_kernel_um', 5.0),
        'predict': {},
    }, indent=2))
    WEIGHT_DIRS.append(d)
    print(f'  -> {k}  ({src})')
assert len(WEIGHT_DIRS) == 2, WEIGHT_DIRS
print('WEIGHT_DIRS =', WEIGHT_DIRS)


In [ ]:
                                      
 
                                                                                             
                                                                               
                                                   
from exp.exp001.io_zarr import dataset_name, find_zarr_datasets
from exp.exp001.submission import write_submission
from exp.exp002.predict import load_model, predict_video

PREDICT_OVERRIDES = {
                                                           
                                                       
    'det_threshold': 0.90,
                                                        
                                                                             
                                                       
                                                                               
                                                                                       
                                                          
                                                                               
                                                                                    
    'linefit_weight': 0.8,
    'linefit_window': 3,
    'pool_kernel_um': 3.0,
    'link_radius_um': 7.0,
    'edge_threshold': 0.5,                                      
    'max_children_per_node': 1,
    'min_track_len': 7,
    'motion_gate_um': None,
    'link_method': 'ilp_flow',
    'ilp_appearance_weight': 0.0,
    'ilp_disappearance_weight': 1.5,
    'ilp_division_weight': 1.0,
    'motion_relink': True,                                                             
                                                                        
                                                                
                                                                             
                                               
                                                
                                                        
                                                        
                                                        
                                                        
                                                                   
                                                        
                                                        
                                                                       
                                                        
                                                        
                                                        
                                                                    
                                                                     
                                                    
                                                            
                                                                  
                                                                     
                                                        
                                                                
                                                                     
                                                        
    'division_forks': True,
    'division_fork_topk': 20,
}

OUT = 'submission.csv'
models, mc = [], None
for d in WEIGHT_DIRS:
    m, cfg = load_model(d)                               
    models.append(m)
    mc = mc or cfg
print(f'models loaded: {len(models)}')
mc.setdefault('predict', {}).update(PREDICT_OVERRIDES)
mc['pool_kernel_um'] = PREDICT_OVERRIDES['pool_kernel_um']                            

zpaths = find_zarr_datasets(TEST_DIR)
assert zpaths, f'.zarr : {TEST_DIR}'
results, skipped = {}, []
for zp in zpaths:
    name = dataset_name(zp)
    try:
        results[name] = predict_video(models, zp, mc)                        
    except Exception as e:                           
        skipped.append((name, f'{type(e).__name__}: {e}'))
        print(f'  !! SKIP {name}: {type(e).__name__}: {e}')
        continue
    print(f'  {name}: nodes={len(results[name].nodes)} edges={len(results[name].edges)}')
if skipped:
    raise RuntimeError(f"Refusing partial submission; failed datasets: {skipped}")
if len(results) != len(zpaths):
    raise RuntimeError(f"Refusing incomplete submission: {len(results)} / {len(zpaths)} datasets")

n_rows = write_submission(results, OUT)
print(f'wrote {n_rows} rows -> {OUT} ({len(results)} datasets, skipped {len(skipped)})')


In [ ]:
                                  
import csv, glob, os

with open('submission.csv') as f:
    r = csv.reader(f); header = next(r); rows = sum(1 for _ in r)
print('header:', header); print('rows  :', rows)

test_names = {os.path.basename(p).removesuffix('.zarr')
              for p in glob.glob(os.path.join(TEST_DIR, '**', '*.zarr'), recursive=True)}
seen = set()
with open('submission.csv') as f:
    r = csv.DictReader(f)
    dcol = 'dataset' if 'dataset' in r.fieldnames else r.fieldnames[0]
    for row in r:
        seen.add(row[dcol])
missing = sorted(test_names - seen)
print(f'datasets: submission {len(seen)} / test {len(test_names)}')
print('!!  dataset:' , missing) if missing else print(' test dataset ')